
### Structured output
Models can be requested to provide their response in a format matching a given schema.
- This is useful for ensuring the output can be easily parsed and used in subsequent processing.
- LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

- It provides FIELD BALIDATION
- DESCRIPTION => Using this, LLM can understand, in which field which data needs to put
- NESTED STRUCTURE

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(model="groq:llama-3.1-8b-instant")
model

e:\GitHub\Python\Python_Agentic_AI_With_Langchain_and_Langraph\practice_agentic_ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000016353E1EE40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000016353E1FB60>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000016353E1EE40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000016353E1FB60>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}}, 'required': ['title', 'year', 'director'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwar

In [5]:
model.invoke("What is the movie Inception about?")

AIMessage(content='Inception is a 2010 science fiction action film written, directed, and produced by Christopher Nolan. The movie follows Cobb (played by Leonardo DiCaprio), a skilled thief who specializes in entering people\'s dreams and stealing their secrets. \n\nCobb is offered a chance to redeem himself by a wealthy businessman named Saito (played by Ken Watanabe). Saito wants Cobb to perform a task known as "inception," which involves planting an idea in someone\'s mind instead of stealing one. \n\nSaito wants Cobb to convince Robert Fischer (played by Cillian Murphy), the son of a dying business magnate, to dissolve his father\'s company. In return, Saito promises to clear Cobb\'s name, which is wanted by the authorities due to the death of his wife.\n\nCobb assembles a team of experts, including his wife Mal\'s twin sister Ariadne (played by Ellen Page), the forger Eames (played by Tom Hardy), the point man Arthur (played by Joseph Gordon-Levitt), and the chemist Yusuf (played

In [9]:
model_with_structure.invoke("What is the movie Inception about?")

Movie(title='Inception', year=2010, director='Christopher Nolan')

## Message output alongside parsed structure

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'wth2xyhv5', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.5,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 290, 'total_tokens': 323, 'completion_time': 0.042098879, 'completion_tokens_details': None, 'prompt_time': 0.018703353, 'prompt_tokens_details': None, 'queue_time': 0.051752797, 'total_time': 0.060802232}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e0cd6-319e-7ac2-bfc3-715ec5be90ea-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}, 'id': 'wth2xyhv5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 290, 'output_tokens': 33, 't

### Nested Structure

In [13]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str | None
    role: str | None

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Dileep Rao', role='Yusuf'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Tom Berenger', role='Browning'), Actor(name='Pete Postlethwaite', role='Maurice Fischer')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)